# 006 — Hybrid Retrieval with Reciprocal Rank Fusion (RRF)
**Storage Series** | Dense + Sparse → best of both worlds

**Reciprocal Rank Fusion** merges ranked lists from multiple retrievers:
`score(d) = Σ 1/(k + rank_i(d))`  where k=60 (standard constant)

This simple formula consistently outperforms any single retriever on BEIR benchmarks.


In [ ]:
!pip install -q rank_bm25

In [ ]:
import os, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=".*LangChain.*")
warnings.filterwarnings('ignore', message=".*allowed_objects.*")
warnings.filterwarnings('ignore', module="langchain.*")
warnings.filterwarnings('ignore', module="langgraph.*")

from dotenv import load_dotenv
load_dotenv(os.path.join(os.getcwd(), '..', '.env'))
GROQ_API_KEY = os.getenv('GROQ_API_KEY')
assert GROQ_API_KEY, "GROQ_API_KEY not found — check ../.env"
print("✓ API key loaded")


In [ ]:
import os, time
os.environ["ANONYMIZED_TELEMETRY"] = "False"  # silence ChromaDB telemetry

from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0, max_retries=2)
llm_smart = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, max_retries=2)
embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    cache_folder=os.path.join(os.getcwd(), '..', 'datasets', 'models'),
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print(f"✓ llm       : {llm.model_name}")
print(f"✓ llm_smart : {llm_smart.model_name}")
print(f"✓ embeddings: all-MiniLM-L6-v2 (384-dim, normalized cosine)")


In [ ]:
import os

DS_DIR = os.path.join(os.getcwd(), '..', 'datasets')
os.makedirs(DS_DIR, exist_ok=True)

def load_doc(fname):
    path = os.path.join(DS_DIR, fname)
    if os.path.exists(path):
        return open(path, encoding='utf-8').read()
    raise FileNotFoundError(
        f"Missing: {path}\n"
        "Run  python create_datasets.py  from the project root."
    )

ai_text  = load_doc("ai.txt")
ml_text  = load_doc("machine_learning.txt")
nlp_text = load_doc("nlp.txt")
dl_text  = load_doc("deep_learning.txt")
rag_text = load_doc("rag.txt")
all_text = "\n\n".join([ai_text, ml_text, nlp_text, dl_text, rag_text])

print(f"✓ 5 dataset files loaded")
print(f"  {'File':<20} {'Chars':>8}  {'~Words':>7}")
print(f"  {'-'*40}")
for name, txt in [('ai.txt', ai_text), ('machine_learning.txt', ml_text),
                  ('nlp.txt', nlp_text), ('deep_learning.txt', dl_text),
                  ('rag.txt', rag_text)]:
    print(f"  {name:<20} {len(txt):>8,}  {len(txt.split()):>7,}")
print(f"  {'-'*40}")
print(f"  {'TOTAL':<20} {len(all_text):>8,}  {len(all_text.split()):>7,}")


In [ ]:
# ── Build both retrievers ────────────────────────────────────────────────
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever

splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40)
chunks = splitter.split_text(all_text[:20000])
docs   = [Document(page_content=c, metadata={"idx": i}) for i, c in enumerate(chunks)]

faiss_store    = FAISS.from_documents(docs, embeddings)
dense_ret      = faiss_store.as_retriever(search_kwargs={"k": 10})
sparse_ret     = BM25Retriever.from_documents(docs, k=10)

print(f"✓ Dense + sparse retrievers over {len(docs)} docs")


In [ ]:
# ── Manual RRF implementation ─────────────────────────────────────────────
def rrf_fuse(ranked_lists: list[list], k: int = 60) -> list:
    '''
    Merge multiple ranked doc lists using Reciprocal Rank Fusion.
    ranked_lists: list of [Document, ...] (each pre-ranked by a retriever)
    Returns: deduplicated documents sorted by RRF score (descending)
    '''
    scores: dict[str, float] = {}
    doc_map: dict[str, object] = {}

    for ranked in ranked_lists:
        for rank, doc in enumerate(ranked, start=1):
            key = doc.page_content[:80]           # use content prefix as ID
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            doc_map[key] = doc

    return [doc_map[k] for k in sorted(scores, key=scores.__getitem__, reverse=True)]

def hybrid_retrieve(query: str, k: int = 5):
    dense_docs  = dense_ret.invoke(query)
    sparse_docs = sparse_ret.invoke(query)
    fused       = rrf_fuse([dense_docs, sparse_docs])
    return fused[:k]

print("✓ RRF hybrid retriever ready")


In [ ]:
# ── LangChain EnsembleRetriever (built-in RRF) ──────────────────────────
from langchain.retrievers import EnsembleRetriever

ensemble = EnsembleRetriever(
    retrievers=[dense_ret, sparse_ret],
    weights=[0.6, 0.4]      # 60% dense, 40% sparse
)
print("✓ EnsembleRetriever (LangChain built-in) ready")


In [ ]:
# ── Side-by-side comparison ──────────────────────────────────────────────
test_queries = [
    "Turing test definition 1950",
    "How does backpropagation work in neural networks?",
    "transformer self-attention mechanism NLP",
]

for q in test_queries:
    dense_top  = dense_ret.invoke(q)[0].page_content[:100]
    sparse_top = sparse_ret.invoke(q)[0].page_content[:100]
    hybrid_top = hybrid_retrieve(q)[0].page_content[:100]
    print(f"Query: {q!r}")
    print(f"  Dense  : {dense_top}...")
    print(f"  Sparse : {sparse_top}...")
    print(f"  Hybrid : {hybrid_top}...")
    print()


In [ ]:
# ── Full hybrid RAG answer ────────────────────────────────────────────────
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    "Answer the question using the retrieved context.\n\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"
)
rag_chain = (
    {"context": lambda q: "\n\n".join(d.page_content for d in hybrid_retrieve(q)),
     "question": lambda q: q}
    | prompt | llm | StrOutputParser()
)

answer = rag_chain.invoke("What is the Turing test and why was it proposed?")
print(answer)


## Key Takeaways
- **RRF formula:** `1/(k + rank)` — simple but remarkably effective
- Dense handles semantic, sparse handles lexical → hybrid covers both
- Weight tuning (0.6/0.4) matters — tune on your domain's eval set
- EnsembleRetriever is a drop-in LangChain solution for production
